In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/Project/training/best_model/model_state_dict.pth" "/content/model_state_dict.pth"

In [ ]:
!cp "/content/drive/MyDrive/Project/data_clean/data_clean_v2.zip" "/content/data_clean.zip"

!unzip -q "/content/data_clean.zip" -d "/content/data_clean/"

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.transforms import v2 as T
import torchvision.transforms.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from tqdm.notebook import tqdm

reports_dir = '/content/reports'
os.makedirs(reports_dir, exist_ok=True)
print(f"Reports directory ensured at: {reports_dir}")

class ResizeAndPad:
    def __init__(self, target_size, fill=0):
        self.target_h, self.target_w = target_size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        if w > self.target_w:
            img.thumbnail((self.target_w, h))
            w, h = img.size
        if h > self.target_h:
            img.thumbnail((w, self.target_h))
            w, h = img.size
        pad_h = max(self.target_h - h, 0)
        pad_w = max(self.target_w - w, 0)
        padding = (pad_w // 2, pad_h // 2, pad_w - pad_w // 2, pad_h - pad_h // 2)
        return F.pad(img, padding, fill=self.fill)

img_size = 150
base_transforms = T.Compose([
    ResizeAndPad((img_size, img_size)),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class DeepFakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

model = DeepFakeDetector()
model.load_state_dict(torch.load('/content/model_state_dict.pth', map_location=device))
model = model.to(device)
model.eval()

test_dir = '/content/data_clean/data_clean/Test'

test_dataset = ImageFolder(root=test_dir, transform=base_transforms)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Class mapping: {test_dataset.class_to_idx}")
print(f"Total test images: {len(test_dataset)}")

all_labels = []
all_probs = []

print("Running evaluation on Test Set...")
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device)
        labels = labels.to(device)


        outputs = model(images)
        probs = torch.sigmoid(outputs).squeeze()

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


all_preds = [1 if p > 0.5 else 0 for p in all_probs]

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\n✅ True Test Accuracy: {test_acc * 100:.2f}%")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake (0)', 'Real (1)'],
            yticklabels=['Fake (0)', 'Real (1)'],
            annot_kws={"size": 14})
plt.title('Confusion Matrix on Test Set', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
cm_path = os.path.join(reports_dir, 'confusion_matrix.png')
plt.savefig(cm_path, bbox_inches='tight')
plt.show()
print(f"Saved Confusion Matrix to {cm_path}")

# ROC Curve
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC)', fontsize=16)
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
roc_path = os.path.join(reports_dir, 'roc_curve.png')
plt.savefig(roc_path, bbox_inches='tight')
plt.show()
print(f"Saved ROC Curve to {roc_path}")

In [ ]:
!cp "/content/reports/confusion_matrix.png" "/content/drive/MyDrive/Project/reports/"

In [ ]:
!cp "/content/reports/roc_curve.png" "/content/drive/MyDrive/Project/reports/"